<a href="https://colab.research.google.com/github/tracyhann/bsl-data-tools/blob/main/REDCap_cleaner_step_by_step.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# REDCap Cleaner

Run each cell from top to bottom. Click the ▶ to execute each step/section.

# Setup (ALWAYS RUN THIS FIRST!) 🏃


*   Click the ▶ button below once, next to **[↪ 3 cells hidden]**, to run all setup codes.
*   Alternatively, run each of the cell: Cleaner engine, Helper functions


In [16]:
# @title Cleaner engine {"display-mode":"form"}
from __future__ import annotations

from dataclasses import dataclass
from io import BytesIO
import re
from typing import Iterable

import pandas as pd



SUBJECT_COLUMN_CANDIDATES = (
    "subid",
    "subject_id",
    "subject",
    "participant_id",
    "record_id",
    "record_subject_id",
)
RESERVED_OUTPUT_COLUMNS = {"IRB", "subid", "raw_subid"}


@dataclass(frozen=True)
class ColumnRule:
    original: str
    keep: bool = True
    rename: str = ""


@dataclass
class CleanResult:
    cleaned: pd.DataFrame
    variable_map: dict[str, pd.DataFrame]
    outliers: pd.DataFrame


@dataclass(frozen=True)
class _Identifier:
    raw: object
    raw_text: str
    compact: str
    subid: str | None
    irb: str | None
    is_outlier: bool = False
    outlier_reason: str = ""


def sanitize_irbs(values: Iterable[object]) -> list[str]:
    """Return non-empty digit-only IRB values, preserving first-seen order."""
    seen: set[str] = set()
    clean: list[str] = []
    for value in values:
        digits = re.sub(r"\D+", "", "" if value is None else str(value))
        if digits and digits not in seen:
            clean.append(digits)
            seen.add(digits)
    return clean


def detect_subject_column(columns: Iterable[str]) -> str | None:
    normalized = {_normalize_column_name(column): str(column) for column in columns}
    for candidate in SUBJECT_COLUMN_CANDIDATES:
        normalized_candidate = _normalize_column_name(candidate)
        if normalized_candidate in normalized:
            return normalized[normalized_candidate]
    return None


def _normalize_column_name(value: object) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(value).strip().lower())


def read_table(filename: str, content: bytes) -> pd.DataFrame:
    suffix = filename.lower().rsplit(".", 1)[-1] if "." in filename else ""
    buffer = BytesIO(content)
    if suffix == "csv":
        return pd.read_csv(buffer)
    if suffix in {"xls", "xlsx"}:
        return pd.read_excel(buffer)
    raise ValueError("Upload must be a .csv, .xls, or .xlsx file.")


def clean_table(
    raw: pd.DataFrame,
    *,
    subject_column: str,
    irbs: Iterable[object],
    column_rules: Iterable[ColumnRule],
    include_outlier_raw_values: Iterable[object] = (),
    drop_empty_rows: bool = False,
) -> CleanResult:
    if subject_column not in raw.columns:
        raise ValueError(f"Subject identifier column not found: {subject_column}")

    irb_values = sanitize_irbs(irbs)
    data = raw.copy()
    if drop_empty_rows:
        data = data.dropna(how="all")

    identifiers = _classify_identifiers(data[subject_column], irb_values)
    include_outliers = {str(value) for value in include_outlier_raw_values}

    rows_to_keep: list[bool] = []
    cleaned_ids: list[str | None] = []
    extracted_irbs: list[str | None] = []
    raw_subids: list[object] = []
    outlier_rows: list[dict[str, object]] = []
    excluded_rows: list[dict[str, object]] = []

    for source_index, ident in zip(data.index, identifiers):
        manually_included = ident.raw_text in include_outliers
        excluded = ident.is_outlier and not manually_included
        rows_to_keep.append(not excluded)
        cleaned_ids.append(ident.subid if ident.subid is not None else ident.compact)
        extracted_irbs.append(ident.irb)
        raw_subids.append(ident.raw)
        if ident.is_outlier:
            outlier_rows.append(
                {
                    "source_row": _source_row_number(source_index),
                    "raw_subid": ident.raw,
                    "cleaned_subid": ident.subid if ident.subid is not None else ident.compact,
                    "included": bool(manually_included),
                    "reason": ident.outlier_reason,
                }
            )
        if excluded:
            excluded_rows.append(
                {
                    "raw_subid": ident.raw,
                    "reason": "excluded from cleaned file; manual reviewed",
                }
            )

    base = pd.DataFrame(
        {
            "IRB": extracted_irbs,
            "subid": cleaned_ids,
            "raw_subid": raw_subids,
        },
        index=data.index,
    )

    mapped_columns, column_map = _apply_column_rules(data, subject_column, column_rules)
    cleaned = pd.concat([base, mapped_columns], axis=1)
    cleaned = cleaned.loc[rows_to_keep].reset_index(drop=True)
    cleaned = _none_for_missing(cleaned)

    variable_map = {
        "column_map": pd.DataFrame(column_map),
        "excluded_subjects": pd.DataFrame(excluded_rows, columns=["raw_subid", "reason"]),
        "outlier_review": pd.DataFrame(
            outlier_rows,
            columns=["source_row", "raw_subid", "included", "reason"],
        ),
    }
    return CleanResult(
        cleaned=cleaned,
        variable_map=variable_map,
        outliers=variable_map["outlier_review"],
    )


def dataframes_to_excel_bytes(sheets: dict[str, pd.DataFrame]) -> bytes:
    buffer = BytesIO()
    with pd.ExcelWriter(buffer, engine="openpyxl") as writer:
        for sheet_name, dataframe in sheets.items():
            dataframe.to_excel(writer, sheet_name=sheet_name[:31], index=False)
    return buffer.getvalue()


def _classify_identifiers(values: pd.Series, irbs: list[str]) -> list[_Identifier]:
    initial = [_clean_identifier(value, irbs) for value in values]
    classified: list[_Identifier] = []
    for ident in initial:
        reasons: list[str] = []
        raw_identifier = ident.raw_text.strip()
        if raw_identifier and len(raw_identifier) < 4:
            reasons.append("raw identifier has fewer than 4 characters")
        if not ident.subid:
            reasons.append("identifier does not match s### or numeric format after cleaning")
        elif not re.fullmatch(r"s\d{3}", ident.subid):
            reasons.append("cleaned identifier does not match expected s### format")
        if reasons:
            classified.append(
                _Identifier(
                    raw=ident.raw,
                    raw_text=ident.raw_text,
                    compact=ident.compact,
                    subid=ident.subid,
                    irb=ident.irb,
                    is_outlier=True,
                    outlier_reason="; ".join(reasons),
                )
            )
        else:
            classified.append(ident)
    return classified


def _clean_identifier(value: object, irbs: list[str]) -> _Identifier:
    if pd.isna(value):
        raw_text = ""
    else:
        raw_text = str(value)
    working = raw_text.strip()
    digit_view = re.sub(r"\D+", "", working)
    found_irb = next((irb for irb in irbs if irb and irb in digit_view), None)
    if found_irb:
        working = re.sub(re.escape(found_irb), "", working)

    compact = re.sub(r"[-_\s]+", "", working).strip()
    compact = re.sub(r"[^A-Za-z0-9]+", "", compact)
    match = re.fullmatch(r"(?i)(?:sub)?s?(\d+)(?:[A-Za-z])?", compact)
    subid = None
    if match:
        digits = match.group(1)
        subid = f"s{digits.zfill(3) if len(digits) < 3 else digits}"
    else:
        digits_match = re.search(r"\d+", compact)
        if digits_match:
            digits = digits_match.group(0)
            subid = f"s{digits.zfill(3) if len(digits) < 3 else digits}"
    return _Identifier(raw=value, raw_text=raw_text, compact=compact, subid=subid, irb=found_irb)


def _apply_column_rules(
    data: pd.DataFrame,
    subject_column: str,
    column_rules: Iterable[ColumnRule],
) -> tuple[pd.DataFrame, list[dict[str, object]]]:
    output = pd.DataFrame(index=data.index)
    map_rows: list[dict[str, object]] = []
    used_names = set(RESERVED_OUTPUT_COLUMNS)

    rules_by_column = {rule.original: rule for rule in column_rules}
    for column in data.columns:
        rule = rules_by_column.get(column, ColumnRule(original=column))
        role = "subject_id" if column == subject_column else ""
        clean_name = _dedupe_column_name((rule.rename or column).strip(), used_names)
        if rule.keep and column != subject_column:
            output[clean_name] = data[column]
            used_names.add(clean_name)
        map_rows.append(
            {
                "original_column": column,
                "clean_column": clean_name if rule.keep and column != subject_column else "",
                "keep": bool(rule.keep and column != subject_column),
                "role": role,
            }
        )
    return output, map_rows


def _dedupe_column_name(name: str, used_names: set[str]) -> str:
    base = name or "unnamed_column"
    candidate = base
    counter = 2
    while candidate in used_names:
        candidate = f"{base}_{counter}"
        counter += 1
    return candidate


def _source_row_number(index: object) -> object:
    if isinstance(index, int):
        return index + 2
    return index


def _none_for_missing(dataframe: pd.DataFrame) -> pd.DataFrame:
    return dataframe.astype(object).where(pd.notna(dataframe), None)


In [17]:
# @title Helper functions {"display-mode":"form"}
from __future__ import annotations

from pathlib import Path
import re

import ipywidgets as widgets
from IPython.display import FileLink, display

try:
    from google.colab import files as colab_files
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
    IN_COLAB = True
except Exception:
    colab_files = None
    IN_COLAB = False


uploaded_filename = ""
uploaded_dataframe = None
configure_ui = None
timepoint_ui = None
review_result = None
clean_result = None
include_outliers = []
timepoint_dictionary = None
cleaned_path = None
map_path = None


VISIT_COLUMN_CANDIDATES = (
    "visit",
    "visits",
    "visit_number",
    "visitnumber",
    "visit_name",
    "event_name",
    "redcap_event_name",
    "timepoint",
    "time_point",
    "administered during visit number",
)
TIMEPOINT_DICTIONARY_COLUMNS = ["visit_column", "source_visit", "timepoint"]


def require_uploaded_dataframe():
    if uploaded_dataframe is None:
        raise ValueError("Run step 1 to upload a REDCap CSV/XLSX export first.")
    return uploaded_dataframe


def require_configure_ui():
    if configure_ui is None:
        raise ValueError("Run step 2 to configure cleaning first.")
    return configure_ui


def run_clean(include_outlier_values=None):
    dataframe = require_uploaded_dataframe()
    config = require_configure_ui()
    return clean_table(
        dataframe,
        subject_column=config.subject_column,
        irbs=config.irbs,
        column_rules=config.column_rules(),
        include_outlier_raw_values=include_outlier_values or [],
        drop_empty_rows=config.drop_empty_rows.value,
    )


def output_stem(filename: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", Path(filename).stem) or "redcap_export"


def detect_visit_column(columns):
    normalized = {_normalize_column_name(column): str(column) for column in columns}
    for candidate in VISIT_COLUMN_CANDIDATES:
        normalized_candidate = _normalize_column_name(candidate)
        if normalized_candidate in normalized:
            return normalized[normalized_candidate]
    for column in columns:
        normalized_column = _normalize_column_name(column)
        if "visit" in normalized_column or "timepoint" in normalized_column:
            return str(column)
    return None


def canonicalize_timepoint(value):
    if value is None or pd.isna(value):
        return ""
    text = re.sub(r"\s+", " ", str(value)).strip()
    if not text:
        return ""

    visit_match = re.search(r"(?i)\b(?:visit|v)\s*0*(\d+)\b", text)
    if visit_match:
        return f"V{int(visit_match.group(1))}"

    numeric_match = re.fullmatch(r"0*(\d+)(?:\.0+)?", text)
    if numeric_match:
        return f"V{int(numeric_match.group(1))}"

    return text


def empty_timepoint_dictionary():
    return pd.DataFrame(columns=TIMEPOINT_DICTIONARY_COLUMNS)


def apply_timepoint_mapping(clean_result, mapping_ui):
    if mapping_ui is None:
        return empty_timepoint_dictionary()

    dictionary = mapping_ui.timepoint_dictionary()
    visit_column = mapping_ui.selected_visit_column
    if not visit_column or dictionary.empty:
        return dictionary

    column_map = clean_result.variable_map.get("column_map", pd.DataFrame())
    match = column_map[column_map["original_column"].astype(str).eq(str(visit_column))]
    if match.empty or not bool(match.iloc[0].get("keep", False)):
        raise ValueError("Selected visits column must be kept in step 2 before timepoint mapping can be applied.")

    clean_column = match.iloc[0].get("clean_column", "")
    if not clean_column or clean_column not in clean_result.cleaned.columns:
        raise ValueError("Selected visits column is not present in the cleaned output.")

    mapping = {
        _timepoint_value_key(row["source_visit"]): row["timepoint"]
        for _, row in dictionary.iterrows()
        if _timepoint_value_key(row["source_visit"])
    }
    clean_result.cleaned = clean_result.cleaned.copy()
    clean_result.cleaned[clean_column] = clean_result.cleaned[clean_column].map(
        lambda value: mapping.get(_timepoint_value_key(value), value)
    )
    return dictionary


def _timepoint_value_key(value):
    if value is None or pd.isna(value):
        return ""
    if isinstance(value, float) and value.is_integer():
        return str(int(value))
    return str(value).strip()


class TimepointMappingUI:
    def __init__(self, dataframe, config):
        self.dataframe = dataframe
        self.config = config
        self.mapping_controls = []
        self.visit_dropdown = widgets.Dropdown(
            description="Visits column",
            options=[""] + [str(column) for column in dataframe.columns],
            layout=widgets.Layout(width="100%"),
            style={"description_width": "130px"},
        )
        detected = detect_visit_column(dataframe.columns)
        if detected:
            self.visit_dropdown.value = detected
        self.mapping_table = widgets.GridBox(
            [],
            layout=widgets.Layout(
                width="100%",
                max_width="none",
                max_height="420px",
                overflow="auto",
                border="1px solid #c9d6dd",
                padding="0",
                grid_template_columns="minmax(260px, 1fr) minmax(150px, 220px) minmax(190px, 280px)",
                align_items="stretch",
            ),
        )
        self.visit_dropdown.observe(self._on_visit_change, names="value")
        self._render_mapping_controls()
        self.ui = widgets.VBox(
            [
                widgets.HTML("<h3>Review visit/timepoint mapping</h3>"),
                widgets.HTML("Unique values from the selected visits column will be canonicalized in the cleaned file. Edit the final timepoint values before running step 5 if needed."),
                self.visit_dropdown,
                self.mapping_table,
            ],
            layout=widgets.Layout(width="100%", max_width="none"),
        )

    @property
    def selected_visit_column(self):
        return self.visit_dropdown.value

    def timepoint_dictionary(self):
        if not self.selected_visit_column:
            return empty_timepoint_dictionary()
        rows = []
        for raw_value, proposed_value, canonical_text in self.mapping_controls:
            rows.append(
                {
                    "visit_column": self.selected_visit_column,
                    "source_visit": raw_value,
                    "timepoint": canonical_text.value.strip() or proposed_value,
                }
            )
        return pd.DataFrame(rows, columns=TIMEPOINT_DICTIONARY_COLUMNS)

    def _on_visit_change(self, change):
        self._render_mapping_controls()

    def _render_mapping_controls(self):
        self.mapping_controls = []
        if not self.selected_visit_column:
            self.mapping_table.children = (
                widgets.HTML("<b>No visits column selected.</b>"),
                widgets.HTML(""),
                widgets.HTML(""),
            )
            return

        values = []
        seen = set()
        for value in self.dataframe[self.selected_visit_column].drop_duplicates().tolist():
            key = _timepoint_value_key(value)
            if not key or key in seen:
                continue
            seen.add(key)
            values.append(value)

        cells = [
            widgets.HTML("<b>Raw visit value</b>"),
            widgets.HTML("<b>Proposed</b>"),
            widgets.HTML("<b>Use in cleaned file</b>"),
        ]
        if not values:
            cells.extend([
                widgets.HTML("No non-empty visit values found."),
                widgets.HTML(""),
                widgets.HTML(""),
            ])
        for raw_value in values:
            proposed = canonicalize_timepoint(raw_value)
            canonical_text = widgets.Text(value=proposed, layout=widgets.Layout(width="100%"))
            raw_label = widgets.HTML(f"<code>{raw_value}</code>")
            proposed_label = widgets.HTML(f"<code>{proposed}</code>")
            self.mapping_controls.append((raw_value, proposed, canonical_text))
            cells.extend([raw_label, proposed_label, canonical_text])
        self.mapping_table.children = tuple(cells)


# REDCap cleaner step-by-step ⛳


*   Click the ▶ button below to execute *each step*



In [18]:
# @title 1. Upload REDCap export 📄 {"vertical-output":true,"display-mode":"form"}

global uploaded_filename, uploaded_dataframe, configure_ui, review_result, clean_result, include_outliers
from google.colab import data_table
data_table.enable_dataframe_formatter()

if not IN_COLAB or colab_files is None:
    raise RuntimeError("This upload cell is intended to run in Google Colab.")

uploaded = colab_files.upload()
if not uploaded:
    raise ValueError("Choose a .csv, .xls, or .xlsx file first.")

uploaded_filename, content = next(iter(uploaded.items()))
uploaded_dataframe = read_table(uploaded_filename, bytes(content))
configure_ui = None
review_result = None
clean_result = None
include_outliers = []

print(f"{uploaded_filename}: {len(uploaded_dataframe)} rows, {len(uploaded_dataframe.columns)} columns.")
rename_map = {
    col: f"col_{i}" for i, col in enumerate(uploaded_dataframe.columns)
}

df_view = uploaded_dataframe.rename(columns=rename_map)
col_map = pd.DataFrame({
    "shortened_column_name": list(rename_map.values()),
    "original_column_name": list(rename_map.keys())
})


display(data_table.DataTable(col_map,num_rows_per_page=5))
data_table.DataTable(df_view,num_rows_per_page=10)


Saving 58807SchatzbergRapid-HAMD_DATA_LABELS_2026-04-28_2108.csv to 58807SchatzbergRapid-HAMD_DATA_LABELS_2026-04-28_2108.csv
58807SchatzbergRapid-HAMD_DATA_LABELS_2026-04-28_2108.csv: 1536 rows, 71 columns.


,shortened_column_name,original_column_name
0,col_0,Record ID
1,col_1,Event Name
2,col_2,Record/Subject ID:
3,col_3,Administered during visit number:
4,col_4,Initials of BSL staff:
...,...,...
66,col_66,17. Insight: Notes
67,col_67,17 a) Insight Severity/Intensity
68,col_68,17 c) Insight score:
69,col_69,GRID HAMD17 Total =


,col_0,col_1,col_2,col_3,col_4,col_5,col_6,col_7,col_8,col_9,...,col_61,col_62,col_63,col_64,col_65,col_66,col_67,col_68,col_69,col_70
0,2,Delegation Log (Arm 3: Delegation Log),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3,Delegation Log (Arm 3: Delegation Log),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4,Delegation Log (Arm 3: Delegation Log),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,5,Delegation Log (Arm 3: Delegation Log),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,6,Delegation Log (Arm 3: Delegation Log),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1531,MDD_18410,Patient Contact Log (Arm 1: Screening),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1532,MDD_18410,Screening (Visit 1) (Arm 1: Screening),NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,...,NaN,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,-12.0,Incomplete
1533,MDD_18428,Patient Contact Log (Arm 1: Screening),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1534,MDD_18428,Screening (Visit 1) (Arm 1: Screening),NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,...,NaN,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,-12.0,Incomplete


In [19]:
# @title 2. Customize configuration ⚙️ {"vertical-output":true,"display-mode":"form"}

global configure_ui


class ConfigureCleaningUI:
    def __init__(self, dataframe):
        self.dataframe = dataframe
        self.column_controls = []
        self.irb_text = widgets.Text(
            description="IRB(s)",
            placeholder="58807, 54909",
            layout=widgets.Layout(width="100%"),
        )
        self.subject_dropdown = widgets.Dropdown(
            description="Subject key identifier column",
            options=[str(column) for column in dataframe.columns],
            layout=widgets.Layout(width="100%"),
            style={"description_width": "190px"},
        )
        detected = detect_subject_column(dataframe.columns)
        if detected:
            self.subject_dropdown.value = detected
        self.drop_empty_rows = widgets.Checkbox(
            value=False,
            description="Drop fully empty rows",
            indent=False,
        )
        self.column_table = widgets.GridBox(
            [],
            layout=widgets.Layout(
                width="100%",
                max_width="none",
                max_height="520px",
                overflow="auto",
                border="1px solid #c9d6dd",
                padding="0",
                grid_template_columns="72px minmax(260px, 1fr) minmax(190px, 280px)",
                align_items="stretch",
            ),
        )
        self._render_column_controls()
        self.ui = widgets.VBox(
            [
                widgets.HTML("<h3>Configure cleaning</h3>"),
                widgets.VBox([self.irb_text, self.subject_dropdown, self.drop_empty_rows]),
                widgets.HTML(f"<b>Column keep / rename</b>: {len(dataframe.columns)} columns loaded."),
                self.column_table,
            ],
            layout=widgets.Layout(width="100%", max_width="none"),
        )

    @property
    def subject_column(self):
        return self.subject_dropdown.value

    @property
    def irbs(self):
        return [value.strip() for value in self.irb_text.value.split(",") if value.strip()]

    def column_rules(self):
        return [
            ColumnRule(original=original, keep=keep.value, rename=rename.value)
            for original, keep, rename in self.column_controls
        ]

    def _render_column_controls(self):
        cells = [
            widgets.HTML("<b>Keep</b>"),
            widgets.HTML("<b>Column</b>"),
            widgets.HTML("<b>Alternative name</b>"),
        ]
        for column in (str(column) for column in self.dataframe.columns):
            keep = widgets.Checkbox(value=True, indent=False, layout=widgets.Layout(width="34px"))
            rename = widgets.Text(placeholder="Optional new name", layout=widgets.Layout(width="100%"))
            label = widgets.HTML(f"<code>{column}</code>")
            self.column_controls.append((column, keep, rename))
            cells.extend([keep, label, rename])
        self.column_table.children = tuple(cells)


configure_ui = ConfigureCleaningUI(require_uploaded_dataframe())
display(configure_ui.ui)


In [20]:
# @title 3. Review subject ID exclusions 🧐 {"vertical-output":true,"display-mode":"form"}

global review_result

review_result = run_clean(include_outlier_values=[])
print(f"Raw subid values are read from selected column: {require_configure_ui().subject_column}")

if review_result.outliers.empty:
    print("No subject ID outliers found.")
else:
    print(f"{len(review_result.outliers)} subject IDs need review. Showing first 100 below.")
    display(review_result.outliers.head(100))


Raw subid values are read from selected column: Record ID
246 subject IDs need review. Showing first 100 below.


,source_row,raw_subid,included,reason
0,2,2,False,raw identifier has fewer than 4 characters
1,3,3,False,raw identifier has fewer than 4 characters
2,4,4,False,raw identifier has fewer than 4 characters
3,5,5,False,raw identifier has fewer than 4 characters
4,6,6,False,raw identifier has fewer than 4 characters
...,...,...,...,...
95,1387,MDD_3698,False,cleaned identifier does not match expected s##...
96,1388,MDD_3698,False,cleaned identifier does not match expected s##...
97,1389,MDD_4460,False,cleaned identifier does not match expected s##...
98,1390,MDD_4460,False,cleaned identifier does not match expected s##...


In [13]:
# @title 4a. Review visit/timepoint mapping 🗓️ {"vertical-output":true,"display-mode":"form"}

global timepoint_ui, timepoint_dictionary

timepoint_ui = TimepointMappingUI(require_uploaded_dataframe(), require_configure_ui())
timepoint_dictionary = timepoint_ui.timepoint_dictionary()
display(timepoint_ui.ui)


In [ ]:
# @title 4b. (OPTIONAL) Manual correction of reviewed subject IDs ✏️ {"vertical-output":true,"display-mode":"form"}

# If step 3 showed subject IDs that should be kept, add their raw_subid values here.
# Example: include_outliers = ["58807 participant 99999"]
include_outliers = []
outliers = input('If step 3 showed subject IDs that should be kept, enter their raw_subid values, separated by \',\' \nExample: MDD_a, MDD_b\nPress \'Enter\' when finished.\n')
include_outliers = outliers.split(',')
include_outliers = [value.strip() for value in include_outliers if value != '']
print('\nManual inclusion of subject IDs: ')
print(include_outliers)


If step 3 showed subject IDs that should be kept, enter their raw_subid values, separated by ',' 
Example: MDD_a, MDD_b
Press 'Enter' when finished.


Manual inclusion of subject IDs: 
[]


In [14]:
# @title 5. Clean and preview 🧹🧼 {"vertical-output":true,"display-mode":"form"}

from IPython.display import HTML, display

global clean_result, timepoint_ui, timepoint_dictionary

if timepoint_ui is None:
    timepoint_ui = TimepointMappingUI(require_uploaded_dataframe(), require_configure_ui())

clean_result = run_clean(include_outlier_values=include_outliers)
timepoint_dictionary = apply_timepoint_mapping(clean_result, timepoint_ui)

print()
display(HTML("""
<h1 style="font-size:16px; margin: 0 0 16px 0;">
  Cleaned data previews:
</h1>
"""))

print(f"{len(clean_result.cleaned)} cleaned rows are ready. Showing first 100 rows.")

print()

display(data_table.DataTable(clean_result.cleaned,num_rows_per_page=10))

print()
display(HTML("""
<h1 style="font-size:16px; margin: 0 0 16px 0;">
  Timepoint dictionary preview:
</h1>
"""))
print(f"{len(timepoint_dictionary)} rows")
print()


display(data_table.DataTable(timepoint_dictionary,num_rows_per_page=10))


print()
display(HTML("""
<h1 style="font-size:16px; margin: 0 0 16px 0;">
  Variable map previews:
</h1>
"""))
for sheet_name, dataframe in clean_result.variable_map.items():
    print()
    display(HTML(f"""
    <h1 style="font-size:16px; margin: 0 0 16px 0;">
      {sheet_name}
    </h1>
    """))
    print(f"{len(dataframe)} rows")
    print()
    display(data_table.DataTable(dataframe,num_rows_per_page=10))


1290 cleaned rows are ready. Showing first 100 rows.



,IRB,subid,raw_subid,VIsit,Record/Subject ID:,Administered during visit number:,Initials of BSL staff:,Todays date:,1. Depression: Notes,1 a) Depression Severity/Intensity,...,15 b) Hypochondriasis Frequency,15 c) Hypochondriasis score:,16. Loss of Weight: Notes,16 a) Loss of Weight (Put answer for EITHER 'A. When rating by history' OR 'B. When rating by actual weight changes' here. Do not rate both).,16 c) Weight loss score:,17. Insight: Notes,17 a) Insight Severity/Intensity,17 c) Insight score:,GRID HAMD17 Total =,Complete?
0,58807,s001,58807_s001,V1,None,Screening (Visit 1),None,None,None,Severe,...,Absent,None,None,No weight loss OR less than 1 lb (.5 kg) loss ...,None,None,Absent,None,None,Complete
1,58807,s001,58807_s001,Patient Contact Log (Arm 2: Baseline/Active St...,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
2,58807,s001,58807_s001,Adverse Events/Regulatory (Arm 2: Baseline/Act...,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
3,58807,s001,58807_s001,V2,58807_s001,Baseline (Visit 2),NB,2021-06-23 12:04,"Feels depression almost every day , had one go...",Moderate,...,Absent,0.0,"none, gained weight still eating.",No weight loss OR less than 1 lb (.5 kg) loss ...,0.0,None,Absent,0.0,22.0,Complete
4,58807,s001,58807_s001,V3,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1285,None,s973,MDD_973,V1,None,None,None,None,None,None,...,None,-1.0,None,None,None,None,None,None,-12.0,Incomplete
1286,None,s983,MDD_983,V1,None,None,None,None,None,None,...,None,-1.0,None,None,None,None,None,None,-12.0,Incomplete
1287,None,s992,MDD_992,Patient Contact Log (Arm 1: Screening),None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
1288,None,s999,MDD_999,Patient Contact Log (Arm 1: Screening),None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None


67 rows



,visit_column,source_visit,timepoint
0,Event Name,Delegation Log (Arm 3: Delegation Log),If you need to change the visit name
1,Event Name,Screening (Visit 1) (Arm 1: Screening),V1
2,Event Name,Patient Contact Log (Arm 2: Baseline/Active St...,Patient Contact Log (Arm 2: Baseline/Active St...
3,Event Name,Adverse Events/Regulatory (Arm 2: Baseline/Act...,Adverse Events/Regulatory (Arm 2: Baseline/Act...
4,Event Name,Baseline (Visit 2) (Arm 2: Baseline/Active Stu...,V2
...,...,...,...
62,Event Name,Treatment Day 5 (Arm 4: Retreatment),Treatment Day 5 (Arm 4: Retreatment)
63,Event Name,1-Week Follow-Up (Arm 4: Retreatment),1-Week Follow-Up (Arm 4: Retreatment)
64,Event Name,Additional TMS Treatment 6 (Arm 2: Baseline/Ac...,Additional TMS Treatment 6 (Arm 2: Baseline/Ac...
65,Event Name,Patient Contact Log (Arm 1: Screening),Patient Contact Log (Arm 1: Screening)


71 rows



,original_column,clean_column,keep,role
0,Record ID,,False,subject_id
1,Event Name,VIsit,True,
2,Record/Subject ID:,Record/Subject ID:,True,
3,Administered during visit number:,Administered during visit number:,True,
4,Initials of BSL staff:,Initials of BSL staff:,True,
...,...,...,...,...
66,17. Insight: Notes,17. Insight: Notes,True,
67,17 a) Insight Severity/Intensity,17 a) Insight Severity/Intensity,True,
68,17 c) Insight score:,17 c) Insight score:,True,
69,GRID HAMD17 Total =,GRID HAMD17 Total =,True,


246 rows



,raw_subid,reason
0,2,excluded from cleaned file; manual reviewed
1,3,excluded from cleaned file; manual reviewed
2,4,excluded from cleaned file; manual reviewed
3,5,excluded from cleaned file; manual reviewed
4,6,excluded from cleaned file; manual reviewed
...,...,...
241,MDD_18410,excluded from cleaned file; manual reviewed
242,MDD_18410,excluded from cleaned file; manual reviewed
243,MDD_18428,excluded from cleaned file; manual reviewed
244,MDD_18428,excluded from cleaned file; manual reviewed


246 rows



,source_row,raw_subid,cleaned_subid,included,reason
0,2,2,s002,False,raw identifier has fewer than 4 characters
1,3,3,s003,False,raw identifier has fewer than 4 characters
2,4,4,s004,False,raw identifier has fewer than 4 characters
3,5,5,s005,False,raw identifier has fewer than 4 characters
4,6,6,s006,False,raw identifier has fewer than 4 characters
...,...,...,...,...,...
241,1533,MDD_18410,s18410,False,cleaned identifier does not match expected s##...
242,1534,MDD_18410,s18410,False,cleaned identifier does not match expected s##...
243,1535,MDD_18428,s18428,False,cleaned identifier does not match expected s##...
244,1536,MDD_18428,s18428,False,cleaned identifier does not match expected s##...


In [16]:
# @title 6. Download cleaned files ⏬ {"vertical-output":true,"display-mode":"form"}

global cleaned_path, map_path

if clean_result is None:
    raise ValueError("Run step 5 to clean and preview the file first.")

stem = output_stem(uploaded_filename)
cleaned_path = Path(f"{stem}_cleaned.xlsx")
map_path = None

if timepoint_dictionary is None:
    timepoint_dictionary = empty_timepoint_dictionary()

column_variable_dictionary = clean_result.variable_map.get(
    "column_map",
    pd.DataFrame(columns=["original_column", "clean_column", "keep", "role"]),
)
excluded_rows = clean_result.variable_map.get(
    "excluded_subjects",
    pd.DataFrame(columns=["raw_subid", "reason"]),
)

cleaned_path.write_bytes(
    dataframes_to_excel_bytes(
        {
            "_raw": uploaded_dataframe,
            "_cleaned": clean_result.cleaned,
            "timepoint_dictionary": timepoint_dictionary,
            "column_variable_dictionary": column_variable_dictionary,
            "excluded_rows": excluded_rows,
        }
    )
)


filename = input(f'\n\n ✏️ Please enter the file name to save this run. \n\nDefault is {stem}_cleaned.xlsx\n')
if len(filename) < 2: filename = cleaned_path
if '.xlsx' not in filename: filename += '.xlsx'
print(f'\n\n📓 Downloading {filename}')


download_path = Path(filename)
download_path.write_bytes(cleaned_path.read_bytes())


if IN_COLAB:
    colab_files.download(str(download_path))
else:
    display(FileLink(str(cleaned_path)))




 ✏️ Please enter the file name to save this run. 

Default is 58807SchatzbergRapid-HAMD_DATA_LABELS_2026-04-28_2108_cleaned.xlsx
MY-DATA


📓 Downloading MY-DATA.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>